In [ ]:
import geopandas as gdp

dataset_for_merging = gpd.read_file("../data/processed/corridor_centerlines_ridge.gpkg") # or "corridor_centerlines_skeletionization.gpkg"

In [ ]:
import geopandas as gpd

original_trajectories = gpd.read_file("/Users/agata/Desktop/thesis/workflow/data/whole_trajectories_in_shallow_waters_cleaned.gpkg")

In [ ]:
import geopandas as gpd
import re


original_gdf = original_trajectories 

dataset_for_merging['traj_id_list'] = dataset_for_merging['string_traj_ids'].apply(lambda x: re.findall(r'\d+', str(x)))

# Explode the dataframe
# This creates a new row for every ID in the list, keeping the 'cluster_id' associated
mapping_df = dataset_for_merging[['cluster_id','traj_id_list']].explode('traj_id_list')

# Ensure the ID types match 
mapping_df['traj_id_list'] = mapping_df['traj_id_list'].astype(str)
original_gdf['traj_id'] = original_gdf['traj_id'].astype(str)

final_gdf = original_gdf.merge(
    mapping_df, 
    left_on='traj_id', 
    right_on='traj_id_list', 
    how='inner'
)

# remove the redundant column from the join
final_gdf = final_gdf.drop(columns=['traj_id_list'])


In [ ]:
cluster_buffer = dataset_for_merging.buffer(10)

In [ ]:
clipped_trajectories = final_gdf.to_crs(cluster_buffer.crs)


In [ ]:
clipped_trajectories = gpd.clip(clipped_trajectories, cluster_buffer)

In [ ]:
clipped_trajectories = clipped_trajectories.explode(index_parts=False)

In [ ]:
clipped_trajectories = clipped_trajectories[~clipped_trajectories.geometry.is_empty]

In [ ]:
import numpy as np
from shapely.geometry import LineString

# convert a LineString into normalized unit vectors
def get_segment_vectors(line):
    
    coords = np.array(line.coords)
    # calculate displacement (dx, dy) for every segment
    diffs = np.diff(coords, axis=0) 
    
    # calculate lengths of segments to normalize them
    lengths = np.linalg.norm(diffs, axis=1)
    
    # Filter out duplicate points to avoid division by zero
    valid = lengths > 0
    unit_vectors = diffs[valid] / lengths[valid, np.newaxis]
    
    return unit_vectors



In [ ]:
def get_ridge_direction(ridge):
    coords = np.array(ridge.coords)
    direction = coords[-1] - coords[0]
    norm = np.linalg.norm(direction)
    if norm == 0:
        return None
    return direction / norm



In [ ]:
def get_relative_consensus_with_angle(ridge, cluster_tracks):
    ridge_vec = get_ridge_direction(ridge)
    if ridge_vec is None:
        return 0.0, 0.0

    projections = []

    for track in cluster_tracks:
        vectors = get_segment_vectors(track)
        if len(vectors) > 0:
            # dot product with ridge direction
            dots = np.dot(vectors, ridge_vec)
            projections.extend(dots)

    if not projections:
        return 0.0, 0.0

    projections = np.array(projections)

    # Mean alignment 
    mean_proj = np.mean(projections)

    # Consensus strength
    R = np.abs(mean_proj)

    # Determine dominant direction
    if mean_proj >= 0:
        direction_vec = ridge_vec
    else:
        direction_vec = -ridge_vec

    # Convert to heading
    mean_angle_rad = np.arctan2(direction_vec[0], direction_vec[1])
    mean_angle_deg = (np.degrees(mean_angle_rad) + 360) % 360

    return float(R), float(mean_angle_deg)

In [ ]:
import numpy as np
from shapely.geometry import LineString, MultiLineString
# Group the clipped trajectories by their cluster ID
grouped = clipped_trajectories.groupby('cluster_id')

results = []

for cluster_id, frame in grouped:
    # Get the $reference ridge/line for this cluster
    reference_line = dataset_for_merging[dataset_for_merging['cluster_id'] == cluster_id].geometry.iloc[0]
    
    # Calculate Relative Consensus
    
    R, mean_angle = get_relative_consensus_with_angle(reference_line, frame['geometry'].tolist())
    
    # Directionality tag 
    tag = ""
    output_geometries = [reference_line]
    
    if R > 0.7:
        geom = reference_line
        tag = "One-Way"
    elif R < 0.3:
        reversed_line = LineString(list(reference_line.coords)[::-1])
        geom = MultiLineString([reference_line, reversed_line])
        tag = "Two-Way"
    else:
        geom = reference_line
        tag = "Manual Review Needed"

    counts = frame['Heuristic_Tide_Necessity'].value_counts()
   
    val_independent = counts.get('Tide Independent', 0)
    # print(val_independent)
    val_dependent = counts.get('Tide Dependent', 0)
    # print(val_dependent)

   

    results.append({
        'cluster': cluster_id,
        'consensus_score': R,
        'avg_heading': mean_angle,
        'type': tag,
        'geometry': geom,
        'unique_vessel_count': frame['boatId'].nunique(),
        'tide_independent_ratio': val_independent/(val_independent+val_dependent),
        'depth': frame['DRVAL2'].mean()  # Assuming DRVAL2 is the depth reading column
    })


In [ ]:
results.to_file("../../data/processed/final.gpkg", driver="GPKG")